In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Analísis de los residuales

In [ ]:
#!/usr/bin/env python3
"""
RICH hit/residual analysis for the simplified AMS-like Geant4 simulation.

Input CSV columns expected:
    eventID,pmtID,x(mm),y(mm),z(mm),energy(MeV),x0(mm),y0(mm),z0(mm)

The script:
  1) recenters PMT hits with respect to the nominal primary trajectory,
  2) calculates the theoretical Cherenkov-ring radius,
  3) builds deltaR = r - R_theory,
  4) fits P_hit with:
         signal      = truncated Gaussian in deltaR,
         central bg  = truncated Lomax in r,
         Rayleigh bg = truncated Gamma in r,
  5) produces three figures:
         recentered_hits.png
         residual_distribution.png
         radial_distribution_phit.png

Edit only the USER SETTINGS section for normal use.
"""

import numpy as np
import pandas as pd
import matplotlib

# Safe for batch/Linux machines without a GUI.
# Comment these two lines if you prefer an interactive backend.
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from scipy.stats import norm, gamma as gamma_dist


# ============================================================
# USER SETTINGS
# ============================================================

INPUT_FILE = "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_10_0GeV_50000ev.csv"
TAG = "proton_10_0GeV"

# Switch between "proton" and "muon"
PARTICLE = "proton"

# None -> read kinetic energy from energy(MeV) in the CSV.
# Otherwise specify the kinetic energy here in MeV.
KINETIC_ENERGY_MEV = None

# ---- AMS-like RICH geometry / optics ----
N_AEROGEL = 1.04814
N_EXPANSION = 1.0

AEROGEL_THICKNESS_MM = 25.0
Z_AEROGEL_DOWNSTREAM_MM = -205.0
L_VACUUM_MM = 469.0

Z_AEROGEL_CENTER_MM = (
    Z_AEROGEL_DOWNSTREAM_MM - AEROGEL_THICKNESS_MM / 2.0
)
Z_DETECTOR_MM = Z_AEROGEL_DOWNSTREAM_MM + L_VACUUM_MM

# Center-emission approximation inside the aerogel
L_AEROGEL_MM = AEROGEL_THICKNESS_MM / 2.0

# ---- Residual fit/plot interval ----
# None for DELTA_R_MIN means use the physical lower limit -R_theory.
DELTA_R_MIN = None
DELTA_R_MAX = 600.0  # mm
DELTA_R_MAX_FIT = 400.0  # mm

# ---- Plot settings ----
N_BINS_RESIDUAL = 160
N_BINS_RADIAL = 160
MAX_HITS_SCATTER = 50000
SCATTER_POINT_SIZE = 3.0

SAVE_PLOTS = True
SHOW_PLOTS = False

OUTPUT_RECENTERED = "recentered_hits_"+TAG+".png"
OUTPUT_RESIDUALS = "residual_distribution_"+TAG+".png"
OUTPUT_RADIAL = "radial_distribution_phit_"+TAG+".png"


# ============================================================
# PARTICLE MASSES [MeV/c^2]
# ============================================================

PARTICLE_MASS_MEV = {
    "proton": 938.2720813,
    "muon": 105.6583755,
}


# ============================================================
# KINEMATICS AND THEORETICAL RING
# ============================================================

def beta_from_kinetic_energy(kinetic_energy_mev, particle):
    particle = particle.lower()

    if particle not in PARTICLE_MASS_MEV:
        raise ValueError(
            f"Unknown particle '{particle}'. "
            "Use 'proton' or 'muon'."
        )

    mass = PARTICLE_MASS_MEV[particle]
    gamma_rel = 1.0 + kinetic_energy_mev / mass

    return np.sqrt(1.0 - 1.0 / gamma_rel**2)


def theoretical_ring_radius(beta):
    """
    Ring radius for a normally incident particle.

    Photon is approximated as being emitted at the center
    of the aerogel radiator.
    """

    if beta * N_AEROGEL <= 1.0:
        raise ValueError(
            "Particle is below Cherenkov threshold in the aerogel."
        )

    theta_aerogel = np.arccos(
        1.0 / (beta * N_AEROGEL)
    )

    # Snell: n_aero sin(theta_aero) =
    #        n_expansion sin(theta_expansion)
    sin_theta_expansion = (
        N_AEROGEL
        / N_EXPANSION
        * np.sin(theta_aerogel)
    )

    if sin_theta_expansion >= 1.0:
        raise ValueError(
            "Total internal reflection in theoretical ring calculation."
        )

    theta_expansion = np.arcsin(
        sin_theta_expansion
    )

    radius = (
        L_AEROGEL_MM * np.tan(theta_aerogel)
        +
        L_VACUUM_MM * np.tan(theta_expansion)
    )

    return radius, theta_aerogel, theta_expansion


# ============================================================
# TRUNCATED PDF COMPONENTS
# ============================================================

def gaussian_signal_pdf(r, rmin, rmax, ring_radius, mu, sigma):
    """
    Signal Gaussian in deltaR = r - R_theory.

    Equivalent to a Gaussian in r centered at
    R_theory + mu.
    """

    center = ring_radius + mu

    normalization = (
        norm.cdf(rmax, loc=center, scale=sigma)
        -
        norm.cdf(rmin, loc=center, scale=sigma)
    )

    if normalization <= 0:
        return np.zeros_like(r)

    pdf = (
        norm.pdf(r, loc=center, scale=sigma)
        / normalization
    )

    return np.where(
        (r >= rmin) & (r <= rmax),
        pdf,
        0.0
    )


def lomax_cdf(r, scale, p):
    """
    CDF corresponding to:
        f(r) proportional to (1 + r/scale)^(-p)
    for r >= 0 and p > 1.
    """

    r = np.asarray(r)

    cdf = 1.0 - (
        1.0 + np.maximum(r, 0.0) / scale
    )**(-(p - 1.0))

    return np.where(r >= 0.0, cdf, 0.0)


def central_lomax_pdf(r, rmin, rmax, scale, p):
    """
    Central detector background.

    Base normalized Lomax-like PDF:
        f(r) = (p-1)/scale * (1+r/scale)^(-p)
    """

    if scale <= 0.0 or p <= 1.0:
        return np.zeros_like(r)

    base = (
        (p - 1.0) / scale
        *
        (1.0 + r / scale)**(-p)
    )

    normalization = (
        lomax_cdf(rmax, scale, p)
        -
        lomax_cdf(rmin, scale, p)
    )

    if normalization <= 0:
        return np.zeros_like(r)

    pdf = base / normalization

    return np.where(
        (r >= rmin) & (r <= rmax),
        pdf,
        0.0
    )


def rayleigh_gamma_pdf(r, rmin, rmax, shape, scale):
    """
    Broad Rayleigh-scattering background represented
    phenomenologically by a Gamma distribution in r.
    """

    if shape <= 0.0 or scale <= 0.0:
        return np.zeros_like(r)

    normalization = (
        gamma_dist.cdf(
            rmax,
            a=shape,
            scale=scale
        )
        -
        gamma_dist.cdf(
            rmin,
            a=shape,
            scale=scale
        )
    )

    if normalization <= 0:
        return np.zeros_like(r)

    pdf = (
        gamma_dist.pdf(
            r,
            a=shape,
            scale=scale
        )
        / normalization
    )

    return np.where(
        (r >= rmin) & (r <= rmax),
        pdf,
        0.0
    )


def phit_components(r, rmin, rmax, ring_radius, pars):
    """
    pars =
      [mu, sigma, f_central, f_rayleigh,
       lambda_central, p_central,
       gamma_shape, gamma_scale]
    """

    (
        mu,
        sigma,
        f_central,
        f_rayleigh,
        lambda_central,
        p_central,
        gamma_shape,
        gamma_scale
    ) = pars

    f_signal = 1.0 - f_central - f_rayleigh

    signal = gaussian_signal_pdf(
        r,
        rmin,
        rmax,
        ring_radius,
        mu,
        sigma
    )

    central = central_lomax_pdf(
        r,
        rmin,
        rmax,
        lambda_central,
        p_central
    )

    rayleigh = rayleigh_gamma_pdf(
        r,
        rmin,
        rmax,
        gamma_shape,
        gamma_scale
    )

    total = (
        f_signal * signal
        +
        f_central * central
        +
        f_rayleigh * rayleigh
    )

    return total, signal, central, rayleigh


# ============================================================
# UNBINNED LIKELIHOOD FIT
# ============================================================

def negative_log_likelihood(
        pars,
        r_data,
        rmin,
        rmax,
        ring_radius
    ):

    (
        mu,
        sigma,
        f_central,
        f_rayleigh,
        lambda_central,
        p_central,
        gamma_shape,
        gamma_scale
    ) = pars

    # Physical/fit-domain checks
    if sigma <= 0.0:
        return 1e100

    if f_central < 0.0 or f_rayleigh < 0.0:
        return 1e100

    if f_central + f_rayleigh >= 0.95:
        return 1e100

    if lambda_central <= 0.0:
        return 1e100

    if p_central <= 1.0:
        return 1e100

    if gamma_shape <= 0.0 or gamma_scale <= 0.0:
        return 1e100

    total, _, _, _ = phit_components(
        r_data,
        rmin,
        rmax,
        ring_radius,
        pars
    )

    if np.any(total <= 0.0) or np.any(~np.isfinite(total)):
        return 1e100

    return -np.sum(np.log(total))


def fit_phit(r_data, rmin, rmax, ring_radius):
    # Starting values motivated by the 5 GeV proton study.
    initial = np.array([
        -0.5,   # mu [mm]
         3.3,   # sigma [mm]
         0.06,  # central fraction
         0.09,  # Rayleigh fraction
        20.0,   # central scale lambda [mm]
         2.0,   # central p
         2.6,   # Gamma shape
       130.0,   # Gamma scale [mm]
    ])

    bounds = [
        (-10.0, 10.0),    # mu
        (0.3, 20.0),      # sigma
        (0.0, 0.50),      # f_central
        (0.0, 0.70),      # f_rayleigh
        (0.5, 300.0),     # central lambda
        (1.01, 15.0),     # central p
        (0.2, 15.0),      # Gamma shape
        (5.0, 800.0),     # Gamma scale
    ]

    result = minimize(
        negative_log_likelihood,
        initial,
        args=(
            r_data,
            rmin,
            rmax,
            ring_radius
        ),
        method="Powell",
        bounds=bounds,
        options={
            "maxiter": 5000,
            "xtol": 1e-6,
            "ftol": 1e-7
        }
    )

    return result


# ============================================================
# CHI2 / NDF
# ============================================================

def component_bin_probability(
        lo,
        hi,
        ring_radius,
        pars
    ):
    (
        mu,
        sigma,
        f_central,
        f_rayleigh,
        lambda_central,
        p_central,
        gamma_shape,
        gamma_scale
    ) = pars

    f_signal = 1.0 - f_central - f_rayleigh

    # The global fit interval is supplied separately in the
    # caller through clipping of lo/hi and normalization.
    return f_signal, f_central, f_rayleigh


def chi2_ndf(
        r_data,
        rmin,
        rmax,
        ring_radius,
        pars,
        nbins
    ):

    counts, edges = np.histogram(
        r_data,
        bins=nbins,
        range=(rmin, rmax)
    )

    (
        mu,
        sigma,
        f_central,
        f_rayleigh,
        lambda_central,
        p_central,
        gamma_shape,
        gamma_scale
    ) = pars

    f_signal = 1.0 - f_central - f_rayleigh
    n = len(r_data)

    expected = np.zeros(nbins)

    # Signal normalization
    signal_center = ring_radius + mu
    sig_norm = (
        norm.cdf(rmax, loc=signal_center, scale=sigma)
        -
        norm.cdf(rmin, loc=signal_center, scale=sigma)
    )

    # Central normalization
    cen_norm = (
        lomax_cdf(
            rmax,
            lambda_central,
            p_central
        )
        -
        lomax_cdf(
            rmin,
            lambda_central,
            p_central
        )
    )

    # Gamma normalization
    gam_norm = (
        gamma_dist.cdf(
            rmax,
            a=gamma_shape,
            scale=gamma_scale
        )
        -
        gamma_dist.cdf(
            rmin,
            a=gamma_shape,
            scale=gamma_scale
        )
    )

    for i in range(nbins):
        lo = edges[i]
        hi = edges[i + 1]

        ps = (
            norm.cdf(
                hi,
                loc=signal_center,
                scale=sigma
            )
            -
            norm.cdf(
                lo,
                loc=signal_center,
                scale=sigma
            )
        ) / sig_norm

        pc = (
            lomax_cdf(
                hi,
                lambda_central,
                p_central
            )
            -
            lomax_cdf(
                lo,
                lambda_central,
                p_central
            )
        ) / cen_norm

        pr = (
            gamma_dist.cdf(
                hi,
                a=gamma_shape,
                scale=gamma_scale
            )
            -
            gamma_dist.cdf(
                lo,
                a=gamma_shape,
                scale=gamma_scale
            )
        ) / gam_norm

        expected[i] = n * (
            f_signal * ps
            +
            f_central * pc
            +
            f_rayleigh * pr
        )

    valid = expected > 5.0

    chi2 = np.sum(
        (counts[valid] - expected[valid])**2
        / expected[valid]
    )

    n_parameters = len(pars)
    ndf = np.count_nonzero(valid) - n_parameters

    return chi2, ndf


# ============================================================
# PLOTS
# ============================================================

def plot_recentered_hits(df_selected, ring_radius):
    n = len(df_selected)

    if n > MAX_HITS_SCATTER:
        plot_df = df_selected.sample(
            MAX_HITS_SCATTER,
            random_state=12345
        )
    else:
        plot_df = df_selected

    fig, ax = plt.subplots(
        figsize=(8, 8)
    )

    ax.scatter(
        plot_df["dx(mm)"],
        plot_df["dy(mm)"],
        s=SCATTER_POINT_SIZE,
        alpha=0.30,
        rasterized=True,
        label="Detected PMT hits"
    )

    # Nominal projected primary trajectory after recentering
    ax.scatter(
        [0.0],
        [0.0],
        marker="x",
        s=100,
        linewidths=2.0,
        label="Nominal primary trajectory"
    )

    ring = plt.Circle(
        (0.0, 0.0),
        ring_radius,
        fill=False,
        linewidth=2.0,
        label=(
            f"Theoretical ring "
            f"R = {ring_radius:.2f} mm"
        )
    )

    ax.add_patch(ring)

    ax.set_xlabel("x - x0 [mm]")
    ax.set_ylabel("y - y0 [mm]")
    ax.set_title(
        f"Recentered hits: {PARTICLE}, "
        f"K = {kinetic_energy_mev:.1f} MeV"
    )
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    ax.legend()

    fig.tight_layout()

    if SAVE_PLOTS:
        fig.savefig(
            OUTPUT_RECENTERED,
            dpi=200
        )

    if SHOW_PLOTS:
        plt.show()

    plt.close(fig)


def plot_residual_distribution(
        residuals,
        delta_min,
        delta_max,
        ring_radius
    ):

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.hist(
        residuals,
        bins=N_BINS_RESIDUAL,
        range=(delta_min, delta_max),
        histtype="step",
        linewidth=1.7
    )

    ax.axvline(
        0.0,
        linestyle="--",
        linewidth=1.3,
        label="Theoretical ring"
    )

    ax.axvline(
        -ring_radius,
        linestyle=":",
        linewidth=1.3,
        label=r"Physical limit: $\Delta R=-R_{\rm theory}$"
    )

    ax.set_xlabel(r"$\Delta R = r-R_{\rm theory}$ [mm]")
    ax.set_ylabel("Number of hits")
    #ax.set_title("Residual distribution")
    ax.set_xlim(delta_min, delta_max)
    ax.set_yscale("log")
    ax.set_ylim(bottom=1e2)
    ax.set_ylim(top=1e6)
    ax.grid(alpha=0.25)
    ax.legend()

    fig.tight_layout()

    if SAVE_PLOTS:
        fig.savefig(
            OUTPUT_RESIDUALS,
            dpi=200
        )

    if SHOW_PLOTS:
        plt.show()

    plt.close(fig)


def plot_radial_distribution_with_phit(
        r_data,
        rmin,
        rmax,
        ring_radius,
        pars,
        chi2,
        ndf
    ):

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.hist(
        r_data,
        bins=N_BINS_RADIAL,
        range=(rmin, rmax),
        density=True,
        histtype="step",
        linewidth=1.7,
        label="Simulation"
    )

    r_grid = np.linspace(
        rmin,
        rmax,
        2500
    )

    total, signal, central, rayleigh = phit_components(
        r_grid,
        rmin,
        rmax,
        ring_radius,
        pars
    )

    (
        mu,
        sigma,
        f_central,
        f_rayleigh,
        lambda_central,
        p_central,
        gamma_shape,
        gamma_scale
    ) = pars

    f_signal = 1.0 - f_central - f_rayleigh

    ax.plot(
        r_grid,
        total,
        linewidth=2.0,
        label=r"$P_{\rm hit}$"
    )

    # Components are useful for interpreting the fitted model.
    ax.plot(
        r_grid,
        f_signal * signal,
        linestyle="--",
        linewidth=1.3,
        label="Ring signal"
    )

    ax.plot(
        r_grid,
        f_central * central,
        linestyle="--",
        linewidth=1.3,
        label="Central background"
    )

    ax.plot(
        r_grid,
        f_rayleigh * rayleigh,
        linestyle="--",
        linewidth=1.3,
        label="Rayleigh background"
    )

    ax.axvline(
        ring_radius,
        linestyle=":",
        linewidth=1.3,
        label=(
            f"$R_{{theory}}$ = "
            f"{ring_radius:.2f} mm"
        )
    )

    ax.set_xlabel("r [mm]")
    ax.set_ylabel("Probability density [1/mm]")

    #if ndf > 0:
    #    fit_quality = chi2 / ndf
    #    title = (
    #        "Radial distribution and "
    #        rf"$P_{{hit}}$  "
    #        rf"$\chi^2/\mathrm{{ndf}}={fit_quality:.2f}$"
    #    )
    #else:
    #    title = "Radial distribution and $P_{hit}$"

    #ax.set_title(title)
    ax.set_xlim(rmin, rmax)
    ax.set_yscale("log")
    ax.set_ylim(bottom=1e-5)
    ax.set_ylim(top=2e-1)
    ax.grid(alpha=0.25)
    ax.legend()

    fig.tight_layout()

    if SAVE_PLOTS:
        fig.savefig(
            OUTPUT_RADIAL,
            dpi=200
        )

    if SHOW_PLOTS:
        plt.show()

    plt.close(fig)


# ============================================================
# MAIN
# ============================================================

df = pd.read_csv(INPUT_FILE)

required_columns = [
    "x(mm)",
    "y(mm)",
    "x0(mm)",
    "y0(mm)",
]

missing = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing)
    )


# ------------------------------------------------------------
# Primary kinetic energy
# ------------------------------------------------------------

if KINETIC_ENERGY_MEV is None:
    if "energy(MeV)" not in df.columns:
        raise ValueError(
            "KINETIC_ENERGY_MEV is None, but "
            "'energy(MeV)' is not present in the CSV."
        )

    kinetic_energy_mev = float(
        df["energy(MeV)"]
        .dropna()
        .median()
    )
else:
    kinetic_energy_mev = float(
        KINETIC_ENERGY_MEV
    )


# ------------------------------------------------------------
# Theoretical ring
# ------------------------------------------------------------

beta = beta_from_kinetic_energy(
    kinetic_energy_mev,
    PARTICLE
)

(
    ring_radius,
    theta_aerogel,
    theta_expansion
) = theoretical_ring_radius(beta)


# ------------------------------------------------------------
# Recenter hits and calculate r and deltaR
# ------------------------------------------------------------

df["dx(mm)"] = (
    df["x(mm)"] - df["x0(mm)"]
)

df["dy(mm)"] = (
    df["y(mm)"] - df["y0(mm)"]
)

df["r(mm)"] = np.sqrt(
    df["dx(mm)"]**2
    +
    df["dy(mm)"]**2
)

df["deltaR(mm)"] = (
    df["r(mm)"] - ring_radius
)


# ------------------------------------------------------------
# Residual/radial analysis range
# ------------------------------------------------------------

if DELTA_R_MIN is None:
    delta_min = -ring_radius
else:
    delta_min = max(
        float(DELTA_R_MIN),
        -ring_radius
    )

delta_max = float(
    DELTA_R_MAX
)

rmin = max(
    0.0,
    ring_radius + delta_min
)

rmax = (
    ring_radius + DELTA_R_MAX_FIT
)

mask = (
    np.isfinite(df["r(mm)"])
    &
    (df["r(mm)"] >= rmin)
    &
    (df["r(mm)"] <= rmax)
)

df_selected = (
    df.loc[mask]
    .copy()
)

r_data = (
    df_selected["r(mm)"]
    .to_numpy(dtype=float)
)

residuals = (
    df["deltaR(mm)"]
    .to_numpy(dtype=float)
)

if len(r_data) == 0:
    raise RuntimeError(
        "No hits remain inside the selected fit range."
    )


# ------------------------------------------------------------
# Fit P_hit
# ------------------------------------------------------------

fit_result = fit_phit(
    r_data,
    rmin,
    rmax,
    ring_radius
)

pars = fit_result.x

(
    mu,
    sigma,
    f_central,
    f_rayleigh,
    lambda_central,
    p_central,
    gamma_shape,
    gamma_scale
) = pars

f_signal = (
    1.0
    - f_central
    - f_rayleigh
)

chi2, ndf = chi2_ndf(
    r_data,
    rmin,
    rmax,
    ring_radius,
    pars,
    N_BINS_RADIAL
)


# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

print()
print("============================================")
print("RICH ANALYSIS")
print("============================================")
print(f"Input file              : {INPUT_FILE}")
print(f"Particle                : {PARTICLE}")
print(f"Kinetic energy          : {kinetic_energy_mev:.6f} MeV")
print(f"beta                    : {beta:.8f}")
print(
    f"theta_aerogel           : "
    f"{np.degrees(theta_aerogel):.6f} deg"
)
print(
    f"theta_expansion         : "
    f"{np.degrees(theta_expansion):.6f} deg"
)
print(f"R_theory                : {ring_radius:.6f} mm")
print()
print("Geometry:")
print(
    f"  aerogel thickness     : "
    f"{AEROGEL_THICKNESS_MM:.3f} mm"
)
print(
    f"  aerogel center z      : "
    f"{Z_AEROGEL_CENTER_MM:.3f} mm"
)
print(
    f"  aerogel downstream z  : "
    f"{Z_AEROGEL_DOWNSTREAM_MM:.3f} mm"
)
print(
    f"  detector plane z      : "
    f"{Z_DETECTOR_MM:.3f} mm"
)
print(
    f"  expansion gap         : "
    f"{L_VACUUM_MM:.3f} mm"
)
print()
print(
    f"Residual fit range      : "
    f"[{delta_min:.3f}, {delta_max:.3f}] mm"
)
print(
    f"Radial fit range        : "
    f"[{rmin:.3f}, {rmax:.3f}] mm"
)
print(f"Hits used               : {len(r_data)}")
print()
print("P_hit fit:")
print(f"  success               : {fit_result.success}")
print(f"  message               : {fit_result.message}")
print(f"  mu                     = {mu:.6f} mm")
print(f"  sigma                  = {sigma:.6f} mm")
print(f"  f_signal               = {f_signal:.6f}")
print(f"  f_central              = {f_central:.6f}")
print(f"  f_rayleigh             = {f_rayleigh:.6f}")
print(f"  lambda_central         = {lambda_central:.6f} mm")
print(f"  p_central              = {p_central:.6f}")
print(f"  gamma_shape            = {gamma_shape:.6f}")
print(f"  gamma_scale            = {gamma_scale:.6f} mm")

if ndf > 0:
    print(
        f"  chi2/ndf               = "
        f"{chi2:.3f}/{ndf} = {chi2/ndf:.3f}"
    )

print("============================================")
print()


# ------------------------------------------------------------
# Produce requested plots
# ------------------------------------------------------------

plot_recentered_hits(
    df_selected,
    ring_radius
)

plot_residual_distribution(
    residuals,
    delta_min,
    delta_max,
    ring_radius
)

plot_radial_distribution_with_phit(
    r_data,
    rmin,
    rmax,
    ring_radius,
    pars,
    chi2,
    ndf
)

if SAVE_PLOTS:
    print("Saved:")
    print(f"  {OUTPUT_RECENTERED}")
    print(f"  {OUTPUT_RESIDUALS}")
    print(f"  {OUTPUT_RADIAL}")


RICH ANALYSIS
Input file              : /content/drive/MyDrive/datos hits/pmt_hits_all_proton_10_0GeV_50000ev.csv
Particle                : proton
Kinetic energy          : 10000.000000 MeV
beta                    : 0.99631420
theta_aerogel           : 16.744198 deg
theta_expansion         : 17.575868 deg
R_theory                : 152.318872 mm

Geometry:
  aerogel thickness     : 25.000 mm
  aerogel center z      : -217.500 mm
  aerogel downstream z  : -205.000 mm
  detector plane z      : 264.000 mm
  expansion gap         : 469.000 mm

Residual fit range      : [-152.319, 600.000] mm
Radial fit range        : [0.000, 552.319] mm
Hits used               : 2865136

P_hit fit:
  success               : True
  message               : Optimization terminated successfully.
  mu                     = -0.688545 mm
  sigma                  = 3.507128 mm
  f_signal               = 0.867540
  f_central              = 0.058403
  f_rayleigh             = 0.074057
  lambda_central         = 15.9

#  Ajuste de los parámetros en función de beta

In [ ]:
#!/usr/bin/env python3
"""
Fit the P_hit model parameters as functions of beta.

Input:
    The fit results obtained at K = 3.5, 4.1, 5.0, 7.0 and 10.0 GeV
    for proton simulations.

Default beta-dependence:
    p(beta) = p_inf + C * (1 - beta^2)

For parameters with no convincing beta dependence:
    p(beta) = constant

By default:
    - f_rayleigh is treated as constant.
    - p_central is treated as constant.
    - f_signal is not independently fit; it is derived from
          f_signal(beta) = 1 - f_central(beta) - f_rayleigh(beta)

Outputs:
    One PNG per parameter and a CSV summary of the fitted functions.

No uncertainties are included because no parameter errors were supplied,
so all fits are unweighted least-squares fits.
"""

import os
import numpy as np
import pandas as pd
import matplotlib

# Safe backend for Linux/batch running.
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# USER SETTINGS
# ============================================================

OUTPUT_DIR = "parameter_beta_fits"

# Proton mass [GeV/c^2]
PROTON_MASS_GEV = 0.9382720813

# Smooth curve range.
# By default this is exactly the simulated beta interval.
N_BETA_POINTS = 500

# If True, also display plots interactively.
SHOW_PLOTS = False

# Fit choices:
#   "linear_q" : p(beta) = p_inf + C*(1-beta^2)
#   "constant" : p(beta) = p0
#
# f_signal is derived separately and therefore is not listed here.
FIT_MODEL = {
    "mu": "linear_q",
    "sigma": "linear_q",
    "f_central": "linear_q",
    "f_rayleigh": "constant",
    "lambda_central": "linear_q",
    "p_central": "constant",
    "gamma_shape": "linear_q",
    "gamma_scale": "linear_q",
}


# ============================================================
# SIMULATION RESULTS
# ============================================================

energy_gev = np.array([
    3.5,
    4.1,
    5.0,
    7.0,
    10.0
], dtype=float)

parameters = {
    "mu": np.array([
        -0.424544,
        -0.500868,
        -0.571188,
        -0.645001,
        -0.688545
    ]),

    "sigma": np.array([
        3.096826,
        3.215494,
        3.323234,
        3.439912,
        3.507128
    ]),

    "f_signal": np.array([
        0.839657,
        0.850816,
        0.858532,
        0.865922,
        0.867540
    ]),

    "f_central": np.array([
        0.087077,
        0.076354,
        0.068359,
        0.060956,
        0.058403
    ]),

    "f_rayleigh": np.array([
        0.073267,
        0.072830,
        0.073109,
        0.073121,
        0.074057
    ]),

    "lambda_central": np.array([
        13.471327,
        14.646727,
        14.808593,
        15.057554,
        15.943443
    ]),

    "p_central": np.array([
        1.676681,
        1.708368,
        1.693834,
        1.698155,
        1.716439
    ]),

    "gamma_shape": np.array([
        2.255629,
        2.282892,
        2.343809,
        2.402072,
        2.392018
    ]),

    "gamma_scale": np.array([
        165.080123,
        165.058736,
        161.121027,
        159.930017,
        159.318582
    ]),
}


# ============================================================
# LABELS / UNITS
# ============================================================

parameter_labels = {
    "mu": r"$\mu$",
    "sigma": r"$\sigma$",
    "f_signal": r"$f_{\rm signal}$",
    "f_central": r"$f_{\rm central}$",
    "f_rayleigh": r"$f_{\rm Rayleigh}$",
    "lambda_central": r"$\lambda_{\rm central}$",
    "p_central": r"$p_{\rm central}$",
    "gamma_shape": r"$k_R$",
    "gamma_scale": r"$\theta_R$",
}

parameter_units = {
    "mu": "mm",
    "sigma": "mm",
    "f_signal": "",
    "f_central": "",
    "f_rayleigh": "",
    "lambda_central": "mm",
    "p_central": "",
    "gamma_shape": "",
    "gamma_scale": "mm",
}


# ============================================================
# KINEMATICS
# ============================================================

def beta_from_kinetic_energy_proton(kinetic_energy_gev):
    """
    Convert proton kinetic energy K [GeV] to beta.
    """
    kinetic_energy_gev = np.asarray(
        kinetic_energy_gev,
        dtype=float
    )

    gamma_rel = (
        1.0
        + kinetic_energy_gev / PROTON_MASS_GEV
    )

    return np.sqrt(
        1.0
        - 1.0 / gamma_rel**2
    )


# ============================================================
# FIT FUNCTIONS
# ============================================================

def linear_q_function(beta, p_inf, C):
    """
    p(beta) = p_inf + C * (1 - beta^2)
    """
    beta = np.asarray(beta, dtype=float)

    return (
        p_inf
        + C * (1.0 - beta**2)
    )


def constant_function(beta, p0):
    """
    p(beta) = p0
    """
    beta = np.asarray(beta, dtype=float)

    return np.full_like(
        beta,
        p0,
        dtype=float
    )


def r_squared(y, y_fit):
    """
    Coefficient of determination R^2.
    """
    y = np.asarray(y, dtype=float)
    y_fit = np.asarray(y_fit, dtype=float)

    ss_res = np.sum(
        (y - y_fit)**2
    )

    ss_tot = np.sum(
        (y - np.mean(y))**2
    )

    if ss_tot == 0.0:
        return np.nan

    return (
        1.0
        - ss_res / ss_tot
    )


def fit_parameter(beta, values, model):
    """
    Unweighted least-squares fit.

    Returns a dictionary containing coefficients,
    predictions at the simulated beta values, R^2,
    and a human-readable equation.
    """

    beta = np.asarray(beta, dtype=float)
    values = np.asarray(values, dtype=float)

    if model == "linear_q":

        q = 1.0 - beta**2

        # y = p_inf + C*q
        design_matrix = np.column_stack([
            np.ones_like(q),
            q
        ])

        p_inf, C = np.linalg.lstsq(
            design_matrix,
            values,
            rcond=None
        )[0]

        fitted = linear_q_function(
            beta,
            p_inf,
            C
        )

        r2 = r_squared(
            values,
            fitted
        )

        equation = (
            f"p(beta) = {p_inf:.8g} "
            f"{C:+.8g}*(1-beta^2)"
        )

        return {
            "model": model,
            "p0": p_inf,
            "C": C,
            "fit_values": fitted,
            "R2": r2,
            "equation": equation,
        }

    elif model == "constant":

        p0 = np.mean(values)

        fitted = constant_function(
            beta,
            p0
        )

        # R^2 is not particularly informative for a
        # mean-only model, so report RMS scatter instead.
        rms = np.sqrt(
            np.mean(
                (values - p0)**2
            )
        )

        equation = (
            f"p(beta) = {p0:.8g}"
        )

        return {
            "model": model,
            "p0": p0,
            "C": np.nan,
            "fit_values": fitted,
            "R2": np.nan,
            "RMS": rms,
            "equation": equation,
        }

    else:
        raise ValueError(
            f"Unknown model '{model}'"
        )


# ============================================================
# PLOTTING
# ============================================================

def plot_parameter(
        name,
        beta,
        values,
        beta_curve,
        curve_values,
        fit_info
    ):

    label = parameter_labels[name]
    unit = parameter_units[name]

    fig, ax = plt.subplots(
        figsize=(7.5, 5.5)
    )

    ax.scatter(
        beta,
        values,
        s=55,
        label="Simulation"
    )

    ax.plot(
        beta_curve,
        curve_values,
        linewidth=2.0,
        label="Fit"
    )

    if unit:
        ylabel = f"{label} [{unit}]"
    else:
        ylabel = label

    ax.set_xlabel(r"$\beta$")
    ax.set_ylabel(ylabel)

    ax.set_title(
        f"{label} as a function of "
        r"$\beta$"
    )

    if fit_info["model"] == "linear_q":
        annotation = (
            r"$p(\beta)=p_\infty+C(1-\beta^2)$"
            "\n"
            rf"$p_\infty={fit_info['p0']:.6g}$"
            "\n"
            rf"$C={fit_info['C']:.6g}$"
            "\n"
            rf"$R^2={fit_info['R2']:.4f}$"
        )

    else:
        annotation = (
            r"$p(\beta)=p_0$"
            "\n"
            rf"$p_0={fit_info['p0']:.6g}$"
            "\n"
            rf"RMS$={fit_info['RMS']:.3g}$"
        )

    ax.text(
        0.04,
        0.96,
        annotation,
        transform=ax.transAxes,
        va="top",
        ha="left"
    )

    ax.grid(alpha=0.25)
    ax.legend()

    # Give a little horizontal margin.
    beta_span = (
        beta.max()
        - beta.min()
    )

    ax.set_xlim(
        beta.min() - 0.05 * beta_span,
        beta.max() + 0.05 * beta_span
    )

    fig.tight_layout()

    output_file = os.path.join(
        OUTPUT_DIR,
        f"{name}_vs_beta.png"
    )

    fig.savefig(
        output_file,
        dpi=220
    )

    if SHOW_PLOTS:
        plt.show()

    plt.close(fig)


# ============================================================
# MAIN
# ============================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

beta = beta_from_kinetic_energy_proton(
    energy_gev
)

beta_curve = np.linspace(
    beta.min(),
    beta.max(),
    N_BETA_POINTS
)

print()
print("==============================================")
print("PROTON P_hit PARAMETER FITS VS BETA")
print("==============================================")
print()

print("Energy [GeV]     beta")
for energy, beta_value in zip(
        energy_gev,
        beta
    ):
    print(
        f"{energy:8.3f}    "
        f"{beta_value:.8f}"
    )

print()

fit_results = {}


# ------------------------------------------------------------
# Fit all independent parameters
# ------------------------------------------------------------

for name, model in FIT_MODEL.items():

    values = parameters[name]

    fit_info = fit_parameter(
        beta,
        values,
        model
    )

    fit_results[name] = fit_info

    if model == "linear_q":

        curve_values = linear_q_function(
            beta_curve,
            fit_info["p0"],
            fit_info["C"]
        )

    else:

        curve_values = constant_function(
            beta_curve,
            fit_info["p0"]
        )

    plot_parameter(
        name,
        beta,
        values,
        beta_curve,
        curve_values,
        fit_info
    )

    print(
        f"{name:20s}: "
        f"{fit_info['equation']}"
    )

    if model == "linear_q":
        print(
            f"{'':20s}  "
            f"R^2 = {fit_info['R2']:.6f}"
        )
    else:
        print(
            f"{'':20s}  "
            f"RMS scatter = "
            f"{fit_info['RMS']:.6g}"
        )


# ------------------------------------------------------------
# f_signal is derived, not independently fit
# ------------------------------------------------------------

fc_fit = fit_results["f_central"]
fr_fit = fit_results["f_rayleigh"]

if fc_fit["model"] == "linear_q":
    fc_curve = linear_q_function(
        beta_curve,
        fc_fit["p0"],
        fc_fit["C"]
    )
    fc_at_data = linear_q_function(
        beta,
        fc_fit["p0"],
        fc_fit["C"]
    )
else:
    fc_curve = constant_function(
        beta_curve,
        fc_fit["p0"]
    )
    fc_at_data = constant_function(
        beta,
        fc_fit["p0"]
    )

if fr_fit["model"] == "linear_q":
    fr_curve = linear_q_function(
        beta_curve,
        fr_fit["p0"],
        fr_fit["C"]
    )
    fr_at_data = linear_q_function(
        beta,
        fr_fit["p0"],
        fr_fit["C"]
    )
else:
    fr_curve = constant_function(
        beta_curve,
        fr_fit["p0"]
    )
    fr_at_data = constant_function(
        beta,
        fr_fit["p0"]
    )

fs_curve = (
    1.0
    - fc_curve
    - fr_curve
)

fs_at_data = (
    1.0
    - fc_at_data
    - fr_at_data
)

fs_values = parameters["f_signal"]

fs_r2 = r_squared(
    fs_values,
    fs_at_data
)

# Because the default f_central is linear_q and
# f_rayleigh is constant, this is also linear in q.
if (
    fc_fit["model"] == "linear_q"
    and fr_fit["model"] == "constant"
):
    fs_p_inf = (
        1.0
        - fc_fit["p0"]
        - fr_fit["p0"]
    )

    fs_C = -fc_fit["C"]

    fs_equation = (
        f"f_signal(beta) = {fs_p_inf:.8g} "
        f"{fs_C:+.8g}*(1-beta^2)"
    )

    fs_fit_info = {
        "model": "linear_q",
        "p0": fs_p_inf,
        "C": fs_C,
        "R2": fs_r2,
        "equation": fs_equation,
    }

else:
    # Generic case: plot the derived numerical curve.
    fs_fit_info = {
        "model": "derived",
        "p0": np.nan,
        "C": np.nan,
        "R2": fs_r2,
        "equation": (
            "f_signal(beta) = "
            "1 - f_central(beta) - "
            "f_rayleigh(beta)"
        ),
    }


# Custom plot for the derived signal fraction
fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)

ax.scatter(
    beta,
    fs_values,
    s=55,
    label="Simulation"
)

ax.plot(
    beta_curve,
    fs_curve,
    linewidth=2.0,
    label="Derived fit"
)

ax.set_xlabel(r"$\beta$")
ax.set_ylabel(r"$f_{\rm signal}$")
ax.set_title(
    r"$f_{\rm signal}$ as a function of $\beta$"
)

annotation = (
    r"$f_{\rm signal}=1-f_{\rm central}-f_{\rm Rayleigh}$"
    "\n"
    rf"$R^2={fs_r2:.4f}$"
)

if fs_fit_info["model"] == "linear_q":
    annotation += (
        "\n"
        rf"$p_\infty={fs_fit_info['p0']:.6g}$"
        "\n"
        rf"$C={fs_fit_info['C']:.6g}$"
    )

ax.text(
    0.04,
    0.96,
    annotation,
    transform=ax.transAxes,
    va="top",
    ha="left"
)

ax.grid(alpha=0.25)
ax.legend()

beta_span = beta.max() - beta.min()

ax.set_xlim(
    beta.min() - 0.05 * beta_span,
    beta.max() + 0.05 * beta_span
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        OUTPUT_DIR,
        "f_signal_vs_beta.png"
    ),
    dpi=220
)

if SHOW_PLOTS:
    plt.show()

plt.close(fig)

fit_results["f_signal"] = fs_fit_info

print(
    f"{'f_signal':20s}: "
    f"{fs_fit_info['equation']}"
)
print(
    f"{'':20s}  "
    f"R^2 = {fs_r2:.6f}"
)


# ------------------------------------------------------------
# Save the simulation points
# ------------------------------------------------------------

points_df = pd.DataFrame({
    "energy_GeV": energy_gev,
    "beta": beta,
})

for name, values in parameters.items():
    points_df[name] = values

points_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "simulation_parameter_values.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Save fit-function summary
# ------------------------------------------------------------

summary_rows = []

ordered_names = [
    "mu",
    "sigma",
    "f_signal",
    "f_central",
    "f_rayleigh",
    "lambda_central",
    "p_central",
    "gamma_shape",
    "gamma_scale",
]

for name in ordered_names:

    info = fit_results[name]

    row = {
        "parameter": name,
        "model": info["model"],
        "p_inf_or_p0": info.get(
            "p0",
            np.nan
        ),
        "C": info.get(
            "C",
            np.nan
        ),
        "R2": info.get(
            "R2",
            np.nan
        ),
        "equation": info.get(
            "equation",
            ""
        ),
    }

    if "RMS" in info:
        row["RMS_scatter"] = info["RMS"]
    else:
        row["RMS_scatter"] = np.nan

    summary_rows.append(row)

summary_df = pd.DataFrame(
    summary_rows
)

summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "parameter_beta_fit_summary.csv"
    ),
    index=False
)


print()
print("==============================================")
print("OUTPUT")
print("==============================================")
print(
    f"Plots and CSV files saved in: "
    f"{OUTPUT_DIR}/"
)
print()
print(
    "Important: these fits are unweighted because "
    "parameter uncertainties were not supplied."
)
print()


PROTON P_hit PARAMETER FITS VS BETA

Energy [GeV]     beta
   3.500    0.97739859
   4.100    0.98250638
   5.000    0.98743844
   7.000    0.99299028
  10.000    0.99631420

mu                  : p(beta) = -0.74332219 +7.0540873*(1-beta^2)
                      R^2 = 0.999094
sigma               : p(beta) = 3.5921833 -10.968066*(1-beta^2)
                      R^2 = 0.999237
f_central           : p(beta) = 0.050931297 +0.7678438*(1-beta^2)
                      R^2 = 0.978847
f_rayleigh          : p(beta) = 0.0732768
                      RMS scatter = 0.000414938
lambda_central      : p(beta) = 16.177903 -55.399428*(1-beta^2)
                      R^2 = 0.887532
p_central           : p(beta) = 1.6986954
                      RMS scatter = 0.0135422
gamma_shape         : p(beta) = 2.4401169 -4.1710624*(1-beta^2)
                      R^2 = 0.944484
gamma_scale         : p(beta) = 157.69341 +175.39571*(1-beta^2)
                      R^2 = 0.906204
f_signal            : f_signal(beta)

# Reconstrucción con el método

In [ ]:
# ============================================================
# RICH beta reconstruction from P_hit likelihood
#
# Updated model:
#   P_hit(r | beta) =
#       f_signal(beta)   * G(deltaR; mu(beta), sigma(beta))
#     + f_central(beta)  * L(r; lambda_central(beta), p_central)
#     + f_rayleigh       * Gamma(r; k_R(beta), theta_R(beta))
#
# where:
#   deltaR = r - R_theory(beta)
#
# Calibration:
#   proton samples at 3.5, 4.1, 5.0, 7.0 and 10.0 GeV
#   with 50000 simulated events per energy.
#
# Geometry:
#   n_aerogel = 1.04814
#   aerogel thickness = 25 mm
#   emission point assumed at the aerogel center
#   expansion gap = 469 mm
# ============================================================

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.stats import norm
from scipy.stats import gamma as gamma_dist


# ============================================================
# Constants
# ============================================================

# Proton mass [GeV/c^2]
PROTON_MASS_GEV = 0.9382720813

# Smallest allowed probability in the likelihood
EPS = 1e-300


# ============================================================
# Proton kinematics
# ============================================================

def getbeta(K_GeV):
    """
    Calculate beta = v/c for a proton from kinetic energy.

    Parameters
    ----------
    K_GeV : float or array-like
        Proton kinetic energy in GeV.

    Returns
    -------
    beta : float or ndarray
        Proton velocity divided by c.
    """

    gamma_rel = (
        1.0
        + np.asarray(K_GeV, dtype=float)
        / PROTON_MASS_GEV
    )

    return np.sqrt(
        1.0
        - 1.0 / gamma_rel**2
    )


# ============================================================
# Configuration
# ============================================================

INPUT_FILE = "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_3_5GeV_10GeV_full.csv"
TAG = "proton_3GeV_15GeV"
# ------------------------------------------------------------
# AMS-like RICH geometry / optical settings
# ------------------------------------------------------------

N_AEROGEL = 1.04814
N_VACUUM = 1.0

AEROGEL_THICKNESS = 25.0       # mm

# Photon emission point approximated at center of aerogel
L_AEROGEL = (
    AEROGEL_THICKNESS / 2.0
)                                  # 12.5 mm

# Downstream aerogel surface -> PMT plane
L_VACUUM = 469.0                  # mm

# Geometry positions, retained for clarity/documentation
Z_AEROGEL_DOWNSTREAM = -205.0     # mm
Z_AEROGEL_CENTER = (
    Z_AEROGEL_DOWNSTREAM
    - AEROGEL_THICKNESS / 2.0
)                                  # -217.5 mm

Z_DETECTOR = (
    Z_AEROGEL_DOWNSTREAM
    + L_VACUUM
)                                  # +264 mm


# ------------------------------------------------------------
# Residual range used in the P_hit calibration
# ------------------------------------------------------------

# The physical lower limit is always deltaR_min = -R_theory(beta).
DELTA_R_MAX = 500.0               # mm


# ------------------------------------------------------------
# Calibration range
# ------------------------------------------------------------

# P_hit(beta) was calibrated with proton samples from 3.5 to 10 GeV.
BETA_MIN = getbeta(3.5)
BETA_MAX = getbeta(10.0)


# ------------------------------------------------------------
# Reconstruction / plotting settings
# ------------------------------------------------------------

MIN_HITS_PER_EVENT = 2
BETA_BOUNDARY_TOLERANCE = 1e-6

N_BETA_HIST_BINS = 60

OUTPUT_BETA_PLOT = "beta_reconstructed_"+TAG+".png"
OUTPUT_LIKELIHOOD_PLOT = "beta_likelihood_example_"+TAG+".png"


# ============================================================
# 1. P_hit parameters versus beta
#
# Variable parameters use:
#
#     p(beta) = p_inf + C * (1 - beta^2)
#
# f_rayleigh and p_central are taken as constants because their
# fitted values were approximately independent of beta.
#
# f_signal is not independent:
#
#     f_signal = 1 - f_central - f_rayleigh
# ============================================================

def phit_parameters(beta):

    q = 1.0 - beta**2

    # --------------------------------------------------------
    # Direct Cherenkov-ring Gaussian
    # --------------------------------------------------------

    # Gaussian mean [mm]
    mu = (
        -0.7433221882
        + 7.054087263 * q
    )

    # Gaussian width [mm]
    sigma = (
        3.592183269
        - 10.96806611 * q
    )


    # --------------------------------------------------------
    # Central detector-material background: Lomax
    # --------------------------------------------------------

    # Central-background fraction
    f_central = (
        0.05093129728
        + 0.7678438022 * q
    )

    # Lomax scale [mm]
    lambda_central = (
        16.17790314
        - 55.39942787 * q
    )

    # Lomax exponent: approximately beta-independent
    p_central = 1.6986954


    # --------------------------------------------------------
    # Rayleigh-scattering background: Gamma
    # --------------------------------------------------------

    # Approximately beta-independent fraction
    f_rayleigh = 0.0732768

    # Gamma shape
    gamma_shape = (
        2.440116855
        - 4.171062363 * q
    )

    # Gamma scale [mm]
    gamma_scale = (
        157.6934116
        + 175.3957136 * q
    )


    # --------------------------------------------------------
    # Signal fraction from normalization
    # --------------------------------------------------------

    f_signal = (
        1.0
        - f_central
        - f_rayleigh
    )

    return (
        mu,
        sigma,
        f_signal,
        f_central,
        f_rayleigh,
        lambda_central,
        p_central,
        gamma_shape,
        gamma_scale
    )


# ============================================================
# 2. Theoretical Cherenkov ring radius
# ============================================================

def ring_radius(beta):
    """
    Calculate theoretical RICH ring radius for a normally
    incident particle.

    Photon emission is approximated at the center of the
    25-mm aerogel radiator.

    R(beta) =
        L_aerogel * tan(theta_aerogel)
        + L_vacuum * tan(theta_vacuum)
    """

    # Below Cherenkov threshold
    if beta * N_AEROGEL <= 1.0:
        return np.nan

    # Cherenkov angle in aerogel
    cos_theta_aero = (
        1.0
        / (beta * N_AEROGEL)
    )

    theta_aero = np.arccos(
        cos_theta_aero
    )

    # Snell refraction:
    # n_aero sin(theta_aero)
    # = n_vac sin(theta_vac)
    sin_theta_vac = (
        N_AEROGEL
        / N_VACUUM
        * np.sin(theta_aero)
    )

    # Numerical / physical safety
    if np.abs(sin_theta_vac) >= 1.0:
        return np.nan

    theta_vac = np.arcsin(
        sin_theta_vac
    )

    R = (
        L_AEROGEL
        * np.tan(theta_aero)
        +
        L_VACUUM
        * np.tan(theta_vac)
    )

    return R


# ============================================================
# 3. Signal PDF:
#    truncated Gaussian in deltaR
# ============================================================

def signal_pdf(r, beta):
    """
    Direct Cherenkov-ring signal.

    Gaussian variable:
        deltaR = r - R_theory(beta)

    The Gaussian is normalized over:
        -R_theory <= deltaR <= DELTA_R_MAX
    """

    (
        mu,
        sigma,
        _,
        _,
        _,
        _,
        _,
        _,
        _
    ) = phit_parameters(beta)

    R = ring_radius(beta)

    if not np.isfinite(R):
        return np.zeros_like(
            np.asarray(r, dtype=float)
        )

    r = np.asarray(
        r,
        dtype=float
    )

    delta_r = r - R

    delta_r_min = -R
    delta_r_max = DELTA_R_MAX

    normalization = (
        norm.cdf(
            delta_r_max,
            loc=mu,
            scale=sigma
        )
        -
        norm.cdf(
            delta_r_min,
            loc=mu,
            scale=sigma
        )
    )

    if normalization <= 0.0:
        return np.zeros_like(r)

    pdf = (
        norm.pdf(
            delta_r,
            loc=mu,
            scale=sigma
        )
        / normalization
    )

    valid = (
        (delta_r >= delta_r_min)
        &
        (delta_r <= delta_r_max)
    )

    return np.where(
        valid,
        pdf,
        0.0
    )


# ============================================================
# 4. Central-background PDF:
#    truncated Lomax-like distribution in r
#
# L(r; lambda, p) =
#
#   (p-1)/lambda
#   ---------------------------  ,
#   (1 + r/lambda)^p
#
# r >= 0, p > 1
# ============================================================

def lomax_cdf(r, scale, p):
    """
    CDF of the Lomax-like form used for the central background.
    """

    r = np.asarray(
        r,
        dtype=float
    )

    positive_r = np.maximum(
        r,
        0.0
    )

    cdf = (
        1.0
        -
        (
            1.0
            + positive_r / scale
        )**(-(p - 1.0))
    )

    return np.where(
        r >= 0.0,
        cdf,
        0.0
    )


def central_pdf(r, beta):
    """
    Central detector/light-guide background.

    The distribution is defined in physical radial coordinate r,
    not in deltaR.
    """

    (
        _,
        _,
        _,
        _,
        _,
        lambda_central,
        p_central,
        _,
        _
    ) = phit_parameters(beta)

    R = ring_radius(beta)

    if not np.isfinite(R):
        return np.zeros_like(
            np.asarray(r, dtype=float)
        )

    r = np.asarray(
        r,
        dtype=float
    )

    r_min = 0.0
    r_max = (
        R + DELTA_R_MAX
    )

    if (
        lambda_central <= 0.0
        or
        p_central <= 1.0
    ):
        return np.zeros_like(r)

    base_pdf = (
        (p_central - 1.0)
        / lambda_central
        *
        (
            1.0
            + r / lambda_central
        )**(-p_central)
    )

    normalization = (
        lomax_cdf(
            r_max,
            lambda_central,
            p_central
        )
        -
        lomax_cdf(
            r_min,
            lambda_central,
            p_central
        )
    )

    if normalization <= 0.0:
        return np.zeros_like(r)

    pdf = (
        base_pdf
        / normalization
    )

    valid = (
        (r >= r_min)
        &
        (r <= r_max)
    )

    return np.where(
        valid,
        pdf,
        0.0
    )


# ============================================================
# 5. Rayleigh-background PDF:
#    truncated Gamma distribution in r
#
# Gamma(r; k, theta) =
#
#   r^(k-1) exp(-r/theta)
#   ----------------------
#   Gamma(k) theta^k
# ============================================================

def rayleigh_pdf(r, beta):
    """
    Broad Rayleigh-scattering background in radial coordinate r.
    """

    (
        _,
        _,
        _,
        _,
        _,
        _,
        _,
        gamma_shape,
        gamma_scale
    ) = phit_parameters(beta)

    R = ring_radius(beta)

    if not np.isfinite(R):
        return np.zeros_like(
            np.asarray(r, dtype=float)
        )

    r = np.asarray(
        r,
        dtype=float
    )

    r_min = 0.0
    r_max = (
        R + DELTA_R_MAX
    )

    if (
        gamma_shape <= 0.0
        or
        gamma_scale <= 0.0
    ):
        return np.zeros_like(r)

    normalization = (
        gamma_dist.cdf(
            r_max,
            a=gamma_shape,
            scale=gamma_scale
        )
        -
        gamma_dist.cdf(
            r_min,
            a=gamma_shape,
            scale=gamma_scale
        )
    )

    if normalization <= 0.0:
        return np.zeros_like(r)

    pdf = (
        gamma_dist.pdf(
            r,
            a=gamma_shape,
            scale=gamma_scale
        )
        / normalization
    )

    valid = (
        (r >= r_min)
        &
        (r <= r_max)
    )

    return np.where(
        valid,
        pdf,
        0.0
    )


# ============================================================
# 6. Complete P_hit
# ============================================================

def p_hit(r, beta):
    """
    Complete hit-probability density:

      P_hit =
          f_signal   * Gaussian(deltaR)
        + f_central  * Lomax(r)
        + f_rayleigh * Gamma(r)
    """

    (
        _,
        _,
        f_signal,
        f_central,
        f_rayleigh,
        _,
        _,
        _,
        _
    ) = phit_parameters(beta)

    S = signal_pdf(
        r,
        beta
    )

    C = central_pdf(
        r,
        beta
    )

    R_bg = rayleigh_pdf(
        r,
        beta
    )

    P = (
        f_signal * S
        +
        f_central * C
        +
        f_rayleigh * R_bg
    )

    return P


# ============================================================
# 7. Event negative log-likelihood
# ============================================================

def event_nll(beta, r_hits):

    R = ring_radius(beta)

    if not np.isfinite(R):
        return 1e100

    # P_hit ahora recibe la coordenada radial física.
    # El componente de señal evalúa internamente:
    #
    #     deltaR = r - R(beta)
    #
    P = p_hit(
        r_hits,
        beta
    )

    # Los hits fuera del soporte P_hit calibrado reciben una
    # penalización por alta probabilidad.

    P = np.clip(
        P,
        EPS,
        None
    )

    return -np.sum(
        np.log(P)
    )


# ============================================================
# 8. Reconstruct beta for one event
# ============================================================

def reconstruct_beta(r_hits):

    result = minimize_scalar(
        event_nll,
        args=(r_hits,),
        method="bounded",
        bounds=(
            BETA_MIN,
            BETA_MAX
        ),
        options={
            "xatol": 1e-10
        }
    )

    return (
        result.x,
        result.fun,
        result.success
    )


# ============================================================
# 9. Read CSV
# ============================================================

df = pd.read_csv(
    INPUT_FILE
)

required_columns = [
    "eventID",
    "x(mm)",
    "y(mm)",
    "x0(mm)",
    "y0(mm)",
    "energy(MeV)"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing_columns)
    )

print(
    "Number of rows =",
    len(df)
)

print(
    "Number of events =",
    df["eventID"].nunique()
)


# ============================================================
# 10. Recenter hit coordinates
# ============================================================

df["dx(mm)"] = (
    df["x(mm)"]
    -
    df["x0(mm)"]
)

df["dy(mm)"] = (
    df["y(mm)"]
    -
    df["y0(mm)"]
)

df["r(mm)"] = np.sqrt(
    df["dx(mm)"]**2
    +
    df["dy(mm)"]**2
)


# ============================================================
# 11. Reconstruct beta event-by-event
# ============================================================

results = []

for event_id, event in df.groupby(
    "eventID"
):

    r_hits = event[
        "r(mm)"
    ].to_numpy(
        dtype=float
    )

    if len(r_hits) < MIN_HITS_PER_EVENT:
        continue

    beta_rec, nll_min, success = (
        reconstruct_beta(
            r_hits
        )
    )

    results.append({
        "eventID": event_id,
        "nHits": len(r_hits),
        "beta_rec": beta_rec,
        "NLL_min": nll_min,
        "fit_success": success
    })


results = pd.DataFrame(
    results
)

if len(results) == 0:
    raise RuntimeError(
        "No events were reconstructed."
    )

print()
print(
    results.head()
)

print()
print(
    "Successfully reconstructed events:",
    results["fit_success"].sum(),
    "/",
    len(results)
)


# ============================================================
# 12. Select successful reconstructions
# ============================================================

good = results[
    results["fit_success"]
].copy()

# ============================================================
# Add true energy and true beta event-by-event
# ============================================================

# Check that every event has only one primary energy
energy_counts_per_event = (
    df.groupby("eventID")["energy(MeV)"]
    .nunique()
)

bad_events = energy_counts_per_event[
    energy_counts_per_event > 1
]

if len(bad_events) > 0:
    raise ValueError(
        "Some events contain more than one primary energy."
    )


# Primary energy for each event
energy_true_by_event = (
    df.groupby("eventID")["energy(MeV)"]
    .first()
)


# Map true energy to reconstructed-event dataframe
good["energy_true_MeV"] = (
    good["eventID"]
    .map(energy_true_by_event)
)

good["energy_true_GeV"] = (
    good["energy_true_MeV"] / 1e3
)


# True beta for each event
good["beta_true"] = getbeta(
    good["energy_true_GeV"].to_numpy()
)


# Reconstruction ratio
good["beta_ratio"] = (
    good["beta_rec"]
    /
    good["beta_true"]
)

if len(good) == 0:
    raise RuntimeError(
        "No successful beta reconstructions."
    )

# ============================================================
# Determine whether sample is monoenergetic
# ============================================================

unique_energies = np.sort(
    good["energy_true_MeV"].unique()
)

MONOENERGETIC = (
    len(unique_energies) == 1
)

if MONOENERGETIC:

    kinetic_energy_true = float(
        unique_energies[0]
    )

    beta_true = float(
        getbeta(
            kinetic_energy_true / 1e3
        )
    )

    print(
        f"True kinetic energy = "
        f"{kinetic_energy_true:.3f} MeV"
    )

    print(
        f"True beta = "
        f"{beta_true:.8f}"
    )

else:

    kinetic_energy_true = np.nan
    beta_true = np.nan

    print(
        "Input contains multiple primary energies."
    )

    print(
        f"Energy range = "
        f"{good['energy_true_GeV'].min():.3f}"
        f" - "
        f"{good['energy_true_GeV'].max():.3f}"
        f" GeV"
    )


# ============================================================
# 14. Reconstruction statistics
# ============================================================

beta_mean = good[
    "beta_rec"
].mean()

beta_median = good[
    "beta_rec"
].median()

beta_std = good[
    "beta_rec"
].std(
    ddof=1
)

n_events = len(results)
n_good = len(good)

beta_sem = (
    beta_std
    / np.sqrt(n_good)
)

beta_q16 = good[
    "beta_rec"
].quantile(
    0.16
)

beta_q84 = good[
    "beta_rec"
].quantile(
    0.84
)

success_fraction = (
    n_good
    / n_events
)

at_lower_bound = np.mean(
    good["beta_rec"]
    <= BETA_MIN
    + BETA_BOUNDARY_TOLERANCE
)

at_upper_bound = np.mean(
    good["beta_rec"]
    >= BETA_MAX
    - BETA_BOUNDARY_TOLERANCE
)

bias = (
    beta_mean
    - beta_true
)

relative_bias = (
    bias
    / beta_true
)

relative_resolution = (
    beta_std
    / beta_true
)


# ============================================================
# 15. Print statistics
# ============================================================

print()
print("----------------------------------------")
print("Reconstructed beta")
print("----------------------------------------")
print(
    f"True kinetic energy = "
    f"{kinetic_energy_true:.3f} MeV"
)
print(
    f"Theoretical beta    = "
    f"{beta_true:.8f}"
)
print(
    f"Mean beta_rec       = "
    f"{beta_mean:.8f}"
)
print(
    f"Median beta_rec     = "
    f"{beta_median:.8f}"
)
print(
    f"Std beta_rec        = "
    f"{beta_std:.8f}"
)
print(
    f"SEM beta_rec        = "
    f"{beta_sem:.8f}"
)
print(
    f"16th percentile     = "
    f"{beta_q16:.8f}"
)
print(
    f"84th percentile     = "
    f"{beta_q84:.8f}"
)
print(
    f"Bias                = "
    f"{bias:.8e}"
)
print(
    f"Relative bias       = "
    f"{relative_bias:.8e}"
)
print(
    f"Relative resolution = "
    f"{relative_resolution:.8e}"
)


# ============================================================
# 16. Reconstruction-summary dataframe
# ============================================================

summary = pd.DataFrame({
    "energy_true_MeV": [
        kinetic_energy_true
    ],

    "beta_true": [
        beta_true
    ],

    "n_events": [
        n_events
    ],

    "n_reconstructed": [
        n_good
    ],

    "success_fraction": [
        success_fraction
    ],

    "mean_beta_rec": [
        beta_mean
    ],

    "median_beta_rec": [
        beta_median
    ],

    "std_beta_rec": [
        beta_std
    ],

    "sem_beta_rec": [
        beta_sem
    ],

    "q16_beta_rec": [
        beta_q16
    ],

    "q84_beta_rec": [
        beta_q84
    ],

    "bias": [
        bias
    ],

    "relative_bias": [
        relative_bias
    ],

    "relative_resolution": [
        relative_resolution
    ],

    "fraction_lower_bound": [
        at_lower_bound
    ],

    "fraction_upper_bound": [
        at_upper_bound
    ]
})


# ============================================================
# 17. Save event-level and summary files
# ============================================================

beta_tag = (
    f"{beta_true:.6f}"
    .replace(".", "p")
)

event_filename = (
    f"beta_reconstruction_"
    f"betaTrue_{beta_tag}.csv"
)

summary_filename = (
    f"beta_summary_"
    f"betaTrue_{beta_tag}.csv"
)

good.to_csv(
    event_filename,
    index=False
)

summary.to_csv(
    summary_filename,
    index=False
)

print()
print(
    "Saved:",
    event_filename
)

print(
    "Saved:",
    summary_filename
)


# ============================================================
# 18. Plot reconstructed-beta distribution
#
# Logarithmic y-axis with:
#   - mean reconstructed beta
#   - median reconstructed beta
#   - theoretical beta
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

ax.hist(
    good["beta_rec"],
    bins=N_BETA_HIST_BINS,
    histtype="step",
    linewidth=2.0,
    log=True,
    label="Reconstructed events"
)

ax.axvline(
    beta_true,
    linestyle="-",
    linewidth=2.0,
    label=(
        rf"Theoretical "
        rf"$\beta={beta_true:.6f}$"
    )
)

ax.axvline(
    beta_mean,
    linestyle="--",
    linewidth=1.8,
    label=(
        rf"Mean "
        rf"$={beta_mean:.6f}$"
    )
)

ax.axvline(
    beta_median,
    linestyle=":",
    linewidth=2.0,
    label=(
        rf"Median "
        rf"$={beta_median:.6f}$"
    )
)

ax.set_xlabel(
    r"$\beta_{\rm rec}$"
)

ax.set_ylabel(
    "Events"
)

ax.set_yscale(
    "log"
)

ax.set_title(
    "RICH likelihood beta reconstruction"
)

ax.grid(
    alpha=0.2
)

ax.legend()

fig.tight_layout()

fig.savefig(
    OUTPUT_BETA_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(
    "Saved:",
    OUTPUT_BETA_PLOT
)


# ============================================================
# 19. Example likelihood scan for one event
# ============================================================

example_event = good.iloc[0][
    "eventID"
]

event = df[
    df["eventID"]
    == example_event
]

r_hits = event[
    "r(mm)"
].to_numpy(
    dtype=float
)

beta_scan = np.linspace(
    BETA_MIN,
    BETA_MAX,
    1000
)

nll_scan = np.array([
    event_nll(
        beta,
        r_hits
    )
    for beta in beta_scan
])

nll_scan -= np.min(
    nll_scan
)

beta_example = good.loc[
    good["eventID"]
    == example_event,
    "beta_rec"
].iloc[0]

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    beta_scan,
    2.0 * nll_scan,
    linewidth=2
)

ax.axvline(
    beta_example,
    linestyle="--",
    label=(
        rf"$\beta_{{rec}}="
        rf"{beta_example:.6f}$"
    )
)

ax.axvline(
    beta_true,
    linestyle=":",
    label=(
        rf"$\beta_{{true}}="
        rf"{beta_true:.6f}$"
    )
)

ax.axhline(
    1.0,
    linestyle=":",
    label=r"$\Delta(-2\ln L)=1$"
)

ax.set_xlabel(
    r"Trial $\beta$"
)

ax.set_ylabel(
    r"$-2\Delta\ln\mathcal{L}$"
)

ax.set_title(
    f"Likelihood scan: event {example_event}"
)

ax.grid(
    alpha=0.2
)

ax.legend()

fig.tight_layout()

fig.savefig(
    OUTPUT_LIKELIHOOD_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(
    "Saved:",
    OUTPUT_LIKELIHOOD_PLOT
)

# ============================================================
# Beta reconstruction ratio vs true energy
# Event-by-event dispersion plot
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

ax.scatter(
    good["energy_true_GeV"],
    good["beta_ratio"],
    s=3,
    alpha=0.1,
    rasterized=True,
    label="Reconstructed events"
)


# Perfect reconstruction reference
ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1.8,
    label=r"$\beta_{\rm rec}/\beta_{\rm true}=1$"
)


ax.set_xlabel(
    "True proton kinetic energy [GeV]"
)

ax.set_ylabel(
    r"$\beta_{\rm rec}/\beta_{\rm true}$"
)

ax.set_title(
    "Event-by-event RICH beta reconstruction"
)

ax.grid(
    alpha=0.3
)

ax.legend()

fig.tight_layout()

fig.savefig(
    "beta_ratio_vs_energy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(
    "Saved: beta_ratio_vs_energy.png"
)

# ============================================================
# Beta reconstructed vs beta true
# Event-by-event closure plot
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)

ax.scatter(
    good["beta_true"],
    good["beta_rec"],
    s=3,
    alpha=0.08,
    rasterized=True,
    label="Reconstructed events"
)


# ------------------------------------------------------------
# Ideal reconstruction: beta_rec = beta_true
# ------------------------------------------------------------

beta_plot_min = min(
    good["beta_true"].min(),
    good["beta_rec"].min()
)

beta_plot_max = max(
    good["beta_true"].max(),
    good["beta_rec"].max()
)

ax.plot(
    [beta_plot_min, beta_plot_max],
    [beta_plot_min, beta_plot_max],
    linestyle="--",
    linewidth=1.8,
    label=r"Ideal: $\beta_{\rm rec}=\beta_{\rm true}$"
)


# ------------------------------------------------------------
# Plot formatting
# ------------------------------------------------------------

ax.set_xlabel(
    r"$\beta_{\rm true}$"
)

ax.set_ylabel(
    r"$\beta_{\rm rec}$"
)

ax.set_title(
    "Event-by-event RICH beta reconstruction"
)

# Same scale in x and y is important for interpreting
# deviations from the diagonal.
ax.set_xlim(
    beta_plot_min,
    beta_plot_max
)

ax.set_ylim(
    beta_plot_min,
    beta_plot_max
)

ax.set_aspect(
    "equal",
    adjustable="box"
)

ax.grid(
    alpha=0.2
)

ax.legend()

fig.tight_layout()

fig.savefig(
    "beta_rec_vs_beta_true.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print(
    "Saved: beta_rec_vs_beta_true.png"
)

Number of rows = 12940850
Number of events = 249905

   eventID  nHits  beta_rec      NLL_min  fit_success
0        0     37  0.978376  1499.171820         True
1        1     55  0.993651   879.799070         True
2        2     49  0.990784   173.992062         True
3        3     65  0.996047   194.316760         True
4        4     40  0.990002   818.328937         True

Successfully reconstructed events: 249886 / 249886
Input contains multiple primary energies.
Energy range = 3.500 - 10.000 GeV

----------------------------------------
Reconstructed beta
----------------------------------------
True kinetic energy = nan MeV
Theoretical beta    = nan
Mean beta_rec       = 0.99087732
Median beta_rec     = 0.99250078
Std beta_rec        = 0.00492972
SEM beta_rec        = 0.00000986
16th percentile     = 0.98525086
84th percentile     = 0.99548851
Bias                = nan
Relative bias       = nan
Relative resolution = nan

Saved: beta_reconstruction_betaTrue_nan.csv
Saved: beta_summ

/tmp/ipykernel_665/2972255568.py:1537: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.tight_layout()


Saved: beta_rec_vs_beta_true.png


In [ ]:
# ============================================================
# Reconstrucción multiequipo (3.8, 4.5, 6.0 y 8.5 GeV)
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# 1. Definición de rutas de tus archivos .csv
# Ajusta el patrón/ruta según donde tengas alojados tus archivos en Drive
FILES_DICT = {
    3.8: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_3_8GeV_50000ev.csv",
    4.5: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_4_5GeV_50000ev.csv",
    6.0: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_6_0GeV_50000ev.csv",
    8.5: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_8_5GeV_50000ev.csv",
}

summaries = []

for k_gev, file_path in FILES_DICT.items():
    if not os.path.exists(file_path):
        print(f"[!] Archivo no encontrado para {k_gev} GeV: {file_path}")
        continue

    print(f"\n---> Procesando energía: {k_gev} GeV...")
    df_temp = pd.read_csv(file_path)

    # Recentrador de coordenadas de impacto
    df_temp["dx(mm)"] = df_temp["x(mm)"] - df_temp["x0(mm)"]
    df_temp["dy(mm)"] = df_temp["y(mm)"] - df_temp["y0(mm)"]
    df_temp["r(mm)"] = np.sqrt(df_temp["dx(mm)"]**2 + df_temp["dy(mm)"]**2)

    # Reconstrucción evento por evento
    event_results = []
    for event_id, event in df_temp.groupby("eventID"):
        r_hits = event["r(mm)"].to_numpy(dtype=float)
        if len(r_hits) < MIN_HITS_PER_EVENT:
            continue

        beta_rec, nll_min, success = reconstruct_beta(r_hits)
        event_results.append({
            "eventID": event_id,
            "nHits": len(r_hits),
            "beta_rec": beta_rec,
            "NLL_min": nll_min,
            "fit_success": success
        })

    res_df = pd.DataFrame(event_results)
    good_events = res_df[res_df["fit_success"]].copy()

    # Cálculo cinemático teórico
    beta_true_val = float(getbeta(k_gev))

    # Métricas estadísticas
    beta_mean = good_events["beta_rec"].mean()
    beta_median = good_events["beta_rec"].median()
    beta_std = good_events["beta_rec"].std(ddof=1)
    n_good = len(good_events)
    beta_sem = beta_std / np.sqrt(n_good) if n_good > 0 else np.nan
    bias = beta_mean - beta_true_val
    rel_bias = bias / beta_true_val
    rel_res = beta_std / beta_true_val

    # Guardar resumen de esta energía
    sum_row = {
        "energy_GeV": k_gev,
        "energy_true_MeV": k_gev * 1000.0,
        "beta_true": beta_true_val,
        "n_events": len(res_df),
        "n_reconstructed": n_good,
        "mean_beta_rec": beta_mean,
        "median_beta_rec": beta_median,
        "std_beta_rec": beta_std,
        "sem_beta_rec": beta_sem,
        "bias": bias,
        "relative_bias": rel_bias,
        "relative_resolution": rel_res
    }
    summaries.append(sum_row)

    # Guardar CSVs individuales por muestra
    tag_beta = f"{beta_true_val:.6f}".replace(".", "p")
    good_events.to_csv(f"beta_reconstruction_betaTrue_{tag_beta}.csv", index=False)
    pd.DataFrame([sum_row]).to_csv(f"beta_summary_betaTrue_{tag_beta}.csv", index=False)

# Consolidar resultados
df_summary = pd.DataFrame(summaries).sort_values("beta_true").reset_index(drop=True)
df_summary["beta_ratio"] = df_summary["mean_beta_rec"] / df_summary["beta_true"]
df_summary["beta_ratio_error"] = df_summary["sem_beta_rec"] / df_summary["beta_true"]

print("\n============================================================")
print("RESUMEN DE RECONSTRUCCIÓN")
print("============================================================")
print(df_summary[["energy_GeV", "beta_true", "mean_beta_rec", "std_beta_rec", "relative_resolution", "relative_bias"]].to_string(index=False))

# ============================================================
# Gráfica: Comparación beta_rec vs beta_true
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Cierre lineal beta_rec vs beta_true
axes[0].errorbar(
    df_summary["beta_true"],
    df_summary["mean_beta_rec"],
    yerr=df_summary["sem_beta_rec"],
    fmt="o",
    capsize=4,
    color="navy",
    label=r"$\langle\beta_{\rm rec}\rangle \pm \mathrm{SEM}$"
)

b_min = df_summary["beta_true"].min() - 0.001
b_max = df_summary["beta_true"].max() + 0.001
b_line = np.linspace(b_min, b_max, 100)

axes[0].plot(b_line, b_line, "r--", linewidth=1.5, label=r"Ideal: $\beta_{\rm rec} = \beta_{\rm true}$")
axes[0].set_xlabel(r"$\beta_{\rm true}$")
axes[0].set_ylabel(r"$\langle\beta_{\rm rec}\rangle$")
axes[0].set_title(r"Cierre de la reconstrucción de $\beta$")
axes[0].grid(alpha=0.3)
axes[0].legend()

# Panel 2: Cociente beta_rec / beta_true
axes[1].errorbar(
    df_summary["energy_GeV"],
    df_summary["beta_ratio"],
    yerr=df_summary["beta_ratio_error"],
    fmt="s",
    capsize=4,
    color="darkgreen",
    label=r"$\langle\beta_{\rm rec}\rangle / \beta_{\rm true}$"
)
axes[1].axhline(1.0, linestyle="--", color="red", linewidth=1.5, label="Cociente ideal = 1.0")
axes[1].set_xlabel("Energía Cinética [GeV]")
axes[1].set_ylabel(r"$\langle\beta_{\rm rec}\rangle / \beta_{\rm true}$")
axes[1].set_title("Cierre relativo vs Energía")
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig("beta_evaluacion_energias.png", dpi=300)
plt.show()


---> Procesando energía: 3.8 GeV...

---> Procesando energía: 4.5 GeV...

---> Procesando energía: 6.0 GeV...

---> Procesando energía: 8.5 GeV...

RESUMEN DE RECONSTRUCCIÓN
 energy_GeV  beta_true  mean_beta_rec  std_beta_rec  relative_resolution  relative_bias
        3.8   0.980198       0.980391      0.001184             0.001208       0.000197
        4.5   0.985004       0.985043      0.000606             0.000615       0.000039
        6.0   0.990814       0.990791      0.000595             0.000601      -0.000023
        8.5   0.995046       0.995007      0.000737             0.000740      -0.000040


# Analísis de Beta

In [ ]:
# ============================================================
# Reconstrucción multiequipo con ajuste KDE / Moda
# Energías: 3.8, 4.5, 6.0 y 8.5 GeV
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# 1. Definición de rutas
FILES_DICT = {
    3.8: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_3_8GeV_50000ev.csv",
    4.5: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_4_5GeV_50000ev.csv",
    6.0: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_6_0GeV_50000ev.csv",
    8.5: "/content/drive/MyDrive/datos hits/pmt_hits_all_proton_8_5GeV_50000ev.csv",
}

summaries = []

for k_gev, file_path in FILES_DICT.items():
    if not os.path.exists(file_path):
        print(f"[!] Archivo no encontrado para {k_gev} GeV: {file_path}")
        continue

    print(f"\n---> Procesando energía: {k_gev} GeV...")
    df_temp = pd.read_csv(file_path)

    # Recentrador de coordenadas de impacto
    df_temp["dx(mm)"] = df_temp["x(mm)"] - df_temp["x0(mm)"]
    df_temp["dy(mm)"] = df_temp["y(mm)"] - df_temp["y0(mm)"]
    df_temp["r(mm)"] = np.sqrt(df_temp["dx(mm)"]**2 + df_temp["dy(mm)"]**2)

    # Reconstrucción evento por evento
    event_results = []
    for event_id, event in df_temp.groupby("eventID"):
        r_hits = event["r(mm)"].to_numpy(dtype=float)
        if len(r_hits) < MIN_HITS_PER_EVENT:
            continue

        beta_rec, nll_min, success = reconstruct_beta(r_hits)
        event_results.append({
            "eventID": event_id,
            "nHits": len(r_hits),
            "beta_rec": beta_rec,
            "NLL_min": nll_min,
            "fit_success": success
        })

    res_df = pd.DataFrame(event_results)
    good_events = res_df[res_df["fit_success"]].copy()

    # Cálculo cinemático teórico
    beta_true_val = float(getbeta(k_gev))
    betas = good_events["beta_rec"].to_numpy()

    # Estimación de Densidad de Kernel (KDE) y extracción de la moda
    kde = gaussian_kde(betas)
    beta_grid = np.linspace(betas.min(), betas.max(), 10000)
    kde_values = kde(beta_grid)
    beta_mode = beta_grid[np.argmax(kde_values)]

    # Métricas estadísticas usando la moda
    beta_std = good_events["beta_rec"].std(ddof=1)
    bias_mode = beta_mode - beta_true_val
    rel_bias_mode = bias_mode / beta_true_val
    rel_res = beta_std / beta_true_val

    sum_row = {
        "energy_GeV": k_gev,
        "energy_true_MeV": k_gev * 1000.0,
        "beta_true": beta_true_val,
        "n_events": len(res_df),
        "n_reconstructed": len(good_events),
        "mode_beta_rec": beta_mode,
        "std_beta_rec": beta_std,
        "bias_mode": bias_mode,
        "relative_bias_mode": rel_bias_mode,
        "relative_resolution": rel_res
    }
    summaries.append(sum_row)

# --------------------------------------------------------
    # Histograma individual sin normalizar (Cuentas reales)
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 5.5))

    # 1. Histograma con cuentas directas
    counts, bins, _ = ax.hist(
        betas,
        bins=N_BETA_HIST_BINS,
        density=False,  # <-- Se desactiva la normalización
        histtype="step",
        linewidth=1.8,
        label="Eventos reconstruidos"
    )

    # 2. Escalamiento de la curva KDE al conteo real
    bin_width = bins[1] - bins[0]
    kde_scaled = kde_values * len(betas) * bin_width

    # 3. Gráficas y líneas de referencia
    ax.plot(beta_grid, kde_scaled, "r-", linewidth=2.0, label="Ajuste KDE")
    ax.axvline(beta_true_val, color="black", linestyle="-", linewidth=1.5, label=rf"$\beta_{{\rm true}} = {beta_true_val:.6f}$")
    ax.axvline(beta_mode, color="red", linestyle="--", linewidth=1.5, label=rf"$\beta_{{\rm mode}} = {beta_mode:.6f}$")

    # 4. Ajuste de etiquetas
    ax.set_xlabel(r"$\beta_{\rm rec}$")
    ax.set_ylabel("Cuentas / Eventos")
    ax.set_title(rf"Distribución de $\beta$ y ajuste KDE ($K = {k_gev}\ \mathrm{{GeV}}$)")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()

    tag_beta = f"{beta_true_val:.6f}".replace(".", "p")
    fig.savefig(f"/content/drive/MyDrive/datos hits/beta_kde_distribution_K_{k_gev}GeV.png", dpi=300)
    plt.close(fig)

# Consolidar resultados
df_summary = pd.DataFrame(summaries).sort_values("beta_true").reset_index(drop=True)
df_summary["beta_ratio_mode"] = df_summary["mode_beta_rec"] / df_summary["beta_true"]

print("\n============================================================")
print("RESUMEN DE RECONSTRUCCIÓN BASADO EN MODA (KDE)")
print("============================================================")
print(df_summary[["energy_GeV", "beta_true", "mode_beta_rec", "bias_mode", "relative_bias_mode"]].to_string(index=False))

# ============================================================
# Gráficas comparativas usando Moda
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: Cierre beta_mode vs beta_true
axes[0].plot(
    df_summary["beta_true"],
    df_summary["mode_beta_rec"],
    "o",
    markersize=7,
    color="navy",
    label=r"$\beta_{\rm mode}\ (\mathrm{KDE})$"
)

b_min = df_summary["beta_true"].min() - 0.001
b_max = df_summary["beta_true"].max() + 0.001
b_line = np.linspace(b_min, b_max, 100)

axes[0].plot(b_line, b_line, "r--", linewidth=1.5, label=r"Ideal: $\beta_{\rm rec} = \beta_{\rm true}$")
axes[0].set_xlabel(r"$\beta_{\rm true}$")
axes[0].set_ylabel(r"$\beta_{\rm mode}$")
axes[0].set_title(r"Cierre de la reconstrucción ($\beta_{\rm mode}$ vs $\beta_{\rm true}$)")
axes[0].grid(alpha=0.3)
axes[0].legend()

# Panel 2: Cociente beta_mode / beta_true vs Energía
axes[1].plot(
    df_summary["energy_GeV"],
    df_summary["beta_ratio_mode"],
    "s-",
    color="darkgreen",
    label=r"$\beta_{\rm mode} / \beta_{\rm true}$"
)
axes[1].axhline(1.0, linestyle="--", color="red", linewidth=1.5, label="Cociente ideal = 1.0")
axes[1].set_xlabel("Energía Cinética [GeV]")
axes[1].set_ylabel(r"$\beta_{\rm mode} / \beta_{\rm true}$")
axes[1].set_title("Cierre relativo (Moda) vs Energía")
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig("/content/drive/MyDrive/datos hits/beta_mode_evaluacion_energias.png", dpi=300)
plt.show()


---> Procesando energía: 3.8 GeV...

---> Procesando energía: 4.5 GeV...

---> Procesando energía: 6.0 GeV...

---> Procesando energía: 8.5 GeV...

RESUMEN DE RECONSTRUCCIÓN BASADO EN MODA (KDE)
 energy_GeV  beta_true  mode_beta_rec  bias_mode  relative_bias_mode
        3.8   0.980198       0.980197  -0.000001           -0.000002
        4.5   0.985004       0.984998  -0.000006           -0.000006
        6.0   0.990814       0.990805  -0.000009           -0.000009
        8.5   0.995046       0.995050   0.000004            0.000004


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

def diagnostico_conjunto_energias(files_dict, n_eventos_muestra=50):
    print("============================================================")
    print(" INICIANDO DIAGNÓSTICO CONJUNTO DE MODELOS DE LIKELIHOOD")
    print("============================================================\n")

    resumen_diagnostico = []

    # Preparar gráfica conjunta para el Paso 3
    fig, ax = plt.subplots(figsize=(9, 6))
    colores = {3.8: 'purple', 4.5: 'blue', 6.0: 'green', 8.5: 'red'}

    for k_gev, file_path in files_dict.items():
        if not os.path.exists(file_path):
            print(f"[!] Omitiendo {k_gev} GeV: Archivo no encontrado.")
            continue

        print(f"---> Analizando energía: {k_gev} GeV")
        df = pd.read_csv(file_path)

        # Recentrador geométrico
        df["dx(mm)"] = df["x(mm)"] - df["x0(mm)"]
        df["dy(mm)"] = df["y(mm)"] - df["y0(mm)"]
        df["r(mm)"] = np.sqrt(df["dx(mm)"]**2 + df["dy(mm)"]**2)

        # Tomar un evento representativo (con un buen número de hits) para análisis escalar
        # En la práctica, se suele sumar el NLL de N eventos, pero un evento robusto
        # es ideal para evaluar la concavidad (Hessiana) y el AIC de manera rápida.
        eventos_validos = df.groupby("eventID").filter(lambda x: len(x) > 15)
        evento_idx = eventos_validos["eventID"].unique()[0]
        r_hits = eventos_validos[eventos_validos["eventID"] == evento_idx]["r(mm)"].to_numpy(dtype=float)

        # ---------------------------------------------------------
        # Paso 1: Convergencia
        # ---------------------------------------------------------
        result = minimize_scalar(
            event_nll, args=(r_hits,), method="bounded",
            bounds=(getbeta(3.5), getbeta(10.0)), options={"xatol": 1e-10}
        )
        beta_opt = result.x
        nll_min = result.fun

        # ---------------------------------------------------------
        # Paso 2: Análisis de Curvatura (Hessiana)
        # ---------------------------------------------------------
        h_step = 1e-5
        nll_plus = event_nll(beta_opt + h_step, r_hits)
        nll_minus = event_nll(beta_opt - h_step, r_hits)
        hessian = (nll_plus - 2*nll_min + nll_minus) / (h_step**2)
        sigma_beta = 1.0 / np.sqrt(hessian) if hessian > 0 else np.nan

        # ---------------------------------------------------------
        # Paso 3: Verosimilitud de Perfil (Agregado a la gráfica)
        # ---------------------------------------------------------
        beta_scan = np.linspace(beta_opt - 0.008, beta_opt + 0.008, 150)
        nll_scan = np.array([event_nll(b, r_hits) for b in beta_scan])
        nll_scan -= nll_min # Normalizar

        color = colores.get(k_gev, 'black')
        ax.plot(beta_scan, 2.0 * nll_scan, linewidth=2, color=color,
                label=rf'{k_gev} GeV ($\beta_{{opt}}={beta_opt:.5f}$)')

        # ---------------------------------------------------------
        # Paso 4: Criterio AIC (Señal vs Extendido)
        # ---------------------------------------------------------
        def nll_signal_only(beta, r_data):
            # Asume que phit_parameters y signal_pdf están en el scope
            mu, sigma, fs, fc, fr, lam, pc, gs, gsc = phit_parameters(beta)
            P = np.clip(signal_pdf(r_data, beta), 1e-300, None)
            return -np.sum(np.log(P))

        aic_signal = 4 + 2 * nll_signal_only(beta_opt, r_hits) # k=2
        aic_extended = 12 + 2 * nll_min # k=6
        delta_aic = aic_extended - aic_signal

        # ---------------------------------------------------------
        # Paso 5: Diagnóstico de Residuos (χ2 reducido global)
        # ---------------------------------------------------------
        r_all = df["r(mm)"].to_numpy(dtype=float)
        counts, edges = np.histogram(r_all, bins=80, range=(0, r_all.max()))
        r_centers = (edges[:-1] + edges[1:]) / 2

        p_exp = p_hit(r_centers, beta_opt)
        p_exp /= np.sum(p_exp)
        exp_counts = p_exp * len(r_all)

        valid = exp_counts > 5.0
        chi2_red = np.nan
        if np.sum(valid) > 6:
            chi2_val = np.sum((counts[valid] - exp_counts[valid])**2 / exp_counts[valid])
            ndf = np.sum(valid) - 6
            chi2_red = chi2_val / ndf

        # Almacenar métricas
        resumen_diagnostico.append({
            "Energía (GeV)": k_gev,
            "Beta Óptimo": beta_opt,
            "Hessiana": hessian,
            "Error (σ_β)": sigma_beta,
            "ΔAIC": delta_aic,
            "χ² Reducido": chi2_red
        })

    # Finalizar gráfica del Paso 3
    ax.axhline(1.0, color='gray', linestyle=':', label=r'$\Delta(-2\ln L)=1$')
    ax.set_xlabel(r'$\beta$ trial')
    ax.set_ylabel(r'$-2\Delta\ln\mathcal{L}$')
    ax.set_title("Verosimilitud de Perfil Conjunta (3.8 a 8.5 GeV)")
    #ax.set_yscale("log")
    ax.set_xlim(0.95, 1.0)
    ax.set_ylim(bottom=0.5)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.savefig("diagnostico_perfil_conjunto.png", dpi=300)
    plt.show()

    # Consolidar DataFrame
    df_resumen = pd.DataFrame(resumen_diagnostico)
    print("\n============================================================")
    print(" RESUMEN DE DIAGNÓSTICO ESTADÍSTICO")
    print("============================================================")
    print(df_resumen.to_string(index=False, float_format="%.5f"))

    return df_resumen

# Ejecución:
# Asumiendo que tu diccionario FILES_DICT ya está definido en memoria
df_resultados = diagnostico_conjunto_energias(FILES_DICT)

 INICIANDO DIAGNÓSTICO CONJUNTO DE MODELOS DE LIKELIHOOD

---> Analizando energía: 3.8 GeV
---> Analizando energía: 4.5 GeV
---> Analizando energía: 6.0 GeV
---> Analizando energía: 8.5 GeV

 RESUMEN DE DIAGNÓSTICO ESTADÍSTICO
 Energía (GeV)  Beta Óptimo       Hessiana  Error (σ_β)        ΔAIC  χ² Reducido
       3.80000      0.97999 14384441.96083      0.00026 -2683.01073   9083.02261
       4.50000      0.98502 18114292.07493      0.00023  -970.35885  75528.99237
       6.00000      0.99103  9271463.08300      0.00033 -3107.01455  39083.75374
       8.50000      0.99493 14244161.02708      0.00026 -3370.03910  15927.55639
